# Merge all sources into one country-year panel

This notebook combines the three cleaned datasets into a single file that everyone can use for Q1 and Q2.

| Input | Made by | Unit |
|---|---|---|
| `data/control_variables.csv` | `ingestion/control_variables.ipynb` (World Bank WDI + WGI) | 217 economies × 2000–2025 |
| `data/ict_country_year.csv` | `analysis/itu_cleaning_script.ipynb` (ITU) | 249 economies/aggregates × 2000–2026 |
| `data/unesco_developing_countries.csv` | `analysis/unesco.ipynb` (UNESCO UIS) | 130 low/middle-income countries × 2000–2024 |

**Outputs**

| File | Contents |
|---|---|
| `data/all_data.csv` | One row per country per year (217 × 26 = 5,642 rows), all variables |
| `data/all_data_codebook.csv` | Every column: which dataset it came from, its source code, description and coverage |

**Design:** the World Bank controls panel is the backbone (it has every country-year, plus region, income group and the `south_asia` flag). ITU and UNESCO are **left-joined** onto it by `iso3` + `year`, so missing data shows up as `NaN`, never as a missing row.

**To reproduce:** *Run All*. Only needs `pandas`.

In [1]:
from pathlib import Path

import pandas as pd

# Works whether Jupyter starts in the repo root or in analysis/
ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_DIR = ROOT / "data"

KEYS = ["iso3", "year"]
DEVELOPING = ["Low income", "Lower middle income", "Upper middle income"]

pd.set_option("display.max_columns", 100)

## 1. Read the three datasets

In [2]:
controls = pd.read_csv(DATA_DIR / "control_variables.csv")
ict = pd.read_csv(DATA_DIR / "ict_country_year.csv")
unesco = pd.read_csv(DATA_DIR / "unesco_developing_countries.csv")

for name, df in [("controls", controls), ("ict", ict), ("unesco", unesco)]:
    print(f"{name:9s} {df.shape[0]:>5,} rows x {df.shape[1]:>2} cols   years {df.year.min()}-{df.year.max()}")

controls  5,642 rows x 32 cols   years 2000-2025
ict       6,028 rows x 11 cols   years 2000-2026
unesco    3,179 rows x 47 cols   years 2000-2024


## 2. Harmonize the ITU data

- ITU calls the key `country_code`; rename it to `iso3`.
- ITU uses `KSO` for Kosovo, the World Bank uses `XKX`.
- ITU includes regional/income **aggregates** (`WLD`, `LDC`, `AFR`, …) and some territories the World Bank doesn't list (Réunion, Taiwan, …). We keep only economies in the World Bank country list, which also drops the aggregates.
- Keep only the years in the panel (drops a few 2026 rows).

In [3]:
ict = ict.rename(columns={"country_code": "iso3"})
ict["iso3"] = ict["iso3"].replace({"KSO": "XKX"})

valid_iso3 = set(controls.iso3)
dropped = ict.loc[~ict.iso3.isin(valid_iso3), ["iso3", "country"]].drop_duplicates()
print(f"Dropping {len(dropped)} ITU codes not in the World Bank country list:")
print(", ".join(f"{r.iso3} ({r.country})" for r in dropped.itertuples()))

ict = ict[ict.iso3.isin(valid_iso3) & ict.year.between(controls.year.min(), controls.year.max())]
ict = ict.drop(columns="country")  # use the World Bank country name
ict.shape

Dropping 35 ITU codes not in the World Bank country list:
AFR (Africa), AIA (Anguilla), ARB (Arab States), ASC (Ascension), ASP (Asia & Pacific), CIS (CIS countries), COK (Cook Islands), EUR (Europe), FLK (Falkland (Malvinas) Is.), GUF (French Guiana), GLP (Guadeloupe), GGY (Guernsey), HI (High income), JEY (Jersey), LLDC (Land Locked Developing Countries (LLDC)), LDC (Least Developed Countries (LDC)), LI (Low income), LMI (Lower middle income), MTQ (Martinique), MYT (Mayotte), MSR (Montserrat), ANT (Neth. Antilles), NIU (Niue), OE (Other Economies), REU (Réunion), SIDS (Small Island Developing States (SIDS)), SHN (St. Helena), SPM (St. Pierre & Miquelon), TWN (Taiwan, Province of China), AMS (The Americas), TKL (Tokelau), UMI (Upper middle income), VAT (Vatican), WLF (Wallis and Futuna), WLD (World)


(5376, 10)

## 3. Harmonize the UNESCO data

UNESCO already uses `iso3`. Its metadata columns (`country`, `wb_region`, `income_group`, `south_asia`) duplicate the controls file, so we check they agree and then drop them.

In [4]:
meta_check = (
    unesco[["iso3", "wb_region", "income_group", "south_asia"]].drop_duplicates()
    .merge(controls[["iso3", "region", "income_group", "south_asia"]].drop_duplicates(), on="iso3", suffixes=("_u", "_c"))
)
assert (meta_check.wb_region == meta_check.region).all()
assert (meta_check.income_group_u == meta_check.income_group_c).all()
assert (meta_check.south_asia_u == meta_check.south_asia_c).all()
assert set(unesco.iso3) <= valid_iso3

unesco = unesco.drop(columns=["country", "wb_region", "income_group", "south_asia"])
unesco.shape

(3179, 43)

## 4. Merge

In [5]:
for name, df in [("controls", controls), ("ict", ict), ("unesco", unesco)]:
    assert not df.duplicated(KEYS).any(), f"duplicate iso3-year keys in {name}"

# No variable name should appear in more than one dataset (other than the keys)
overlap = (set(controls) & set(ict)) | (set(controls) & set(unesco)) | (set(ict) & set(unesco))
assert overlap == set(KEYS), overlap

all_data = (
    controls
    .merge(ict, on=KEYS, how="left", validate="one_to_one")
    .merge(unesco, on=KEYS, how="left", validate="one_to_one")
)
assert len(all_data) == len(controls)

# Convenience flag: low + lower-middle + upper-middle income (today's classification)
all_data.insert(all_data.columns.get_loc("south_asia") + 1, "developing", all_data.income_group.isin(DEVELOPING))

all_data = all_data.sort_values(KEYS).reset_index(drop=True)
print(f"{len(all_data):,} rows ({all_data.iso3.nunique()} countries x {all_data.year.nunique()} years), {all_data.shape[1]} columns")
all_data.head()

5,642 rows (217 countries x 26 years), 82 columns


,iso3,country,region,income_group,lending_type,south_asia,developing,year,gdp_pc_ppp,gdp_pc_usd,gdp_growth,agri_employment_share,remittances_gdp,electricity_access,electricity_access_rural,population,pop_0_14_share,pop_growth,urban_share,pop_density,fertility_rate,adolescent_fertility,under5_mortality,life_expectancy,edu_spend_gdp,edu_spend_govt,compulsory_edu_years,youth_unemployment,female_lfp,wgi_gov_effectiveness,wgi_political_stability,wgi_rule_of_law,wgi_control_corruption,households_computer_pct,computer_users_pct,fixed_telephone_subscriptions,mobile_cellular_per100,telecom_investment_pct_gdp,telecom_investment_pct_revenue,internet_users_pct,households_internet_estimated_pct,compl_primary,compl_lowersec,compl_uppersec,compl_primary_mod,compl_lowersec_mod,compl_uppersec_mod,oos_rate_primary,oos_rate_lowersec,oos_rate_uppersec,net_enrol_primary,net_enrol_lowersec,net_enrol_uppersec,min_prof_read_primary,min_prof_math_primary,min_prof_read_lowersec,min_prof_math_lowersec,school_internet_primary,school_internet_lowersec,school_internet_uppersec,school_computers_lowersec,school_electricity_lowersec,edu_spend_pct_gdp,qualified_teachers_lowersec,compl_lowersec_poorest,compl_lowersec_richest,compl_lowersec_f,compl_lowersec_m,compl_uppersec_f,compl_uppersec_m,compl_uppersec_mod_f,compl_uppersec_mod_m,compl_uppersec_gpia,oos_rate_uppersec_f,oos_rate_uppersec_m,tertiary_enrol_f,tertiary_enrol_m,youth_literacy_f,youth_literacy_m,attain_uppersec_25plus_f,attain_uppersec_25plus_m,adult_literacy_f
0,ABW,Aruba,Latin America & Caribbean,High income,Not classified,False,False,2000,37618.992616,28104.895436,7.622921,NaN,0.057850,91.7,100.0,90588.0,23.447103,1.030817,65.354550,503.266667,1.845,43.729,NaN,72.939,4.71468,21.807711,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38100.0,16.899314,NaN,NaN,15.442823,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABW,Aruba,Latin America & Caribbean,High income,Not classified,False,False,2001,38827.467274,29007.738696,4.182002,NaN,0.070405,100.0,100.0,91439.0,23.165938,0.935033,65.335114,507.994444,1.813,40.463,NaN,73.044,4.79898,21.323179,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37132.0,58.689995,NaN,NaN,17.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABW,Aruba,Latin America & Caribbean,High income,Not classified,False,False,2002,38195.317608,28535.463948,-0.944953,NaN,0.067773,100.0,100.0,92074.0,22.824844,0.692052,65.282069,511.522222,1.800,38.185,NaN,73.135,4.87220,19.757151,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37132.0,67.630418,NaN,NaN,18.800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABW,Aruba,Latin America & Caribbean,High income,Not classified,False,False,2003,38182.392526,28525.807706,1.110505,NaN,0.013392,100.0,100.0,93128.0,22.456939,1.138229,65.198292,517.377778,1.808,37.807,NaN,73.236,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37390.0,75.779439,NaN,NaN,20.800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABW,Aruba,Latin America & Caribbean,High income,Not classified,False,False,2004,40101.787613,29959.774819,7.293728,NaN,0.064666,100.0,100.0,95138.0,22.076438,2.135358,65.088653,528.544444,1.819,38.761,NaN,73.223,4.35699,14.902910,NaN,NaN,NaN,0.866112,0.623389,0.611885,0.966986,NaN,NaN,38003.0,105.552874,NaN,NaN,23.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Combined codebook

Stacks the three codebooks into one table and recomputes coverage on the merged panel, both for all countries and for developing countries only.

In [6]:
id_cols = pd.DataFrame([
    ("iso3", "World Bank", "", "ISO 3166-1 alpha-3 country code (merge key)"),
    ("country", "World Bank", "", "Country name (World Bank spelling)"),
    ("region", "World Bank", "", "World Bank region (current). Afghanistan and Pakistan are NOT in South Asia here; use south_asia"),
    ("income_group", "World Bank", "", "World Bank income group (today's classification, applied to every year)"),
    ("lending_type", "World Bank", "", "World Bank lending category (IDA / IBRD / Blend)"),
    ("south_asia", "Project", "", "True for the traditional 8 South Asian countries (AFG BGD BTN IND LKA MDV NPL PAK)"),
    ("developing", "Project", "", "True if income_group is Low, Lower middle or Upper middle income"),
    ("year", "", "", "Year"),
], columns=["column", "dataset", "source_code", "description"])

cb_controls = pd.read_csv(DATA_DIR / "control_variables_codebook.csv").rename(columns={"wb_code": "source_code"})
cb_controls = cb_controls.assign(dataset="World Bank " + cb_controls.source)[["column", "dataset", "source_code", "description"]]

cb_ict = pd.read_csv(DATA_DIR / "ict_codebook.csv").dropna(subset=["itu_code"])
cb_ict = pd.DataFrame({
    "column": cb_ict.variable_name, "dataset": "ITU", "source_code": cb_ict.itu_code,
    "description": cb_ict.indicator_name + " [" + cb_ict.unit + "]",
})

cb_unesco = pd.read_csv(DATA_DIR / "unesco_codebook.csv")
cb_unesco = pd.DataFrame({
    "column": cb_unesco.variable, "dataset": "UNESCO UIS", "source_code": cb_unesco.indicator_code,
    "description": cb_unesco.description.str.strip(),
})

codebook = pd.concat([id_cols, cb_controls, cb_ict, cb_unesco], ignore_index=True)

# Every column must be documented, and every documented column must exist
assert set(codebook.column) == set(all_data.columns), set(codebook.column) ^ set(all_data.columns)
codebook = codebook.set_index("column").loc[all_data.columns].rename_axis("column").reset_index()

dev = all_data[all_data.developing]
codebook["pct_nonmissing"] = [round(all_data[c].notna().mean() * 100, 1) for c in codebook.column]
codebook["pct_nonmissing_developing"] = [round(dev[c].notna().mean() * 100, 1) for c in codebook.column]
codebook["first_year"] = [all_data.loc[all_data[c].notna(), "year"].min() for c in codebook.column]
codebook["last_year"] = [all_data.loc[all_data[c].notna(), "year"].max() for c in codebook.column]
codebook

,column,dataset,source_code,description,pct_nonmissing,pct_nonmissing_developing,first_year,last_year
0,iso3,World Bank,,ISO 3166-1 alpha-3 country code (merge key),100.0,100.0,2000,2025
1,country,World Bank,,Country name (World Bank spelling),100.0,100.0,2000,2025
2,region,World Bank,,World Bank region (current). Afghanistan and P...,100.0,100.0,2000,2025
3,income_group,World Bank,,World Bank income group (today's classificatio...,100.0,100.0,2000,2025
4,lending_type,World Bank,,World Bank lending category (IDA / IBRD / Blend),100.0,100.0,2000,2025
...,...,...,...,...,...,...,...,...
77,youth_literacy_f,UNESCO UIS,LR.AG15T24.F,"Youth literacy rate, population 15-24 years, f...",17.4,28.9,2000,2024
78,youth_literacy_m,UNESCO UIS,LR.AG15T24.M,"Youth literacy rate, population 15-24 years, m...",16.3,26.9,2000,2024
79,attain_uppersec_25plus_f,UNESCO UIS,EA.3T8.AG25T99.F,"Educational attainment rate, completed upper s...",18.0,29.8,2000,2024
80,attain_uppersec_25plus_m,UNESCO UIS,EA.3T8.AG25T99.M,"Educational attainment rate, completed upper s...",17.7,29.4,2000,2024


## 6. Save

In [7]:
all_data.to_csv(DATA_DIR / "all_data.csv", index=False)
codebook.to_csv(DATA_DIR / "all_data_codebook.csv", index=False)
print(f"Saved data/all_data.csv ({all_data.shape[0]:,} x {all_data.shape[1]}) and data/all_data_codebook.csv")

Saved data/all_data.csv (5,642 x 82) and data/all_data_codebook.csv


## 7. Sanity check: key variables for each question

Coverage (% of country-years with data) for **developing countries, 2010–2024**.

In [8]:
key_vars = {
    "Q1": ["internet_users_pct", "households_internet_estimated_pct", "compl_primary", "compl_lowersec",
           "compl_lowersec_mod", "oos_rate_primary", "gdp_pc_ppp", "electricity_access"],
    "Q2": ["female_lfp", "compl_uppersec_f", "compl_uppersec_mod_f", "attain_uppersec_25plus_f",
           "tertiary_enrol_f", "youth_literacy_f", "adolescent_fertility", "fertility_rate"],
}
recent_dev = all_data[all_data.developing & all_data.year.between(2010, 2024)]
pd.DataFrame(
    [(q, v, round(recent_dev[v].notna().mean() * 100, 1)) for q, vs in key_vars.items() for v in vs],
    columns=["question", "column", "pct_nonmissing_dev_2010_24"],
)

,question,column,pct_nonmissing_dev_2010_24
0,Q1,internet_users_pct,93.2
1,Q1,households_internet_estimated_pct,85.0
2,Q1,compl_primary,33.4
3,Q1,compl_lowersec,34.5
4,Q1,compl_lowersec_mod,87.0
5,Q1,oos_rate_primary,65.2
6,Q1,gdp_pc_ppp,95.2
7,Q1,electricity_access,99.2
8,Q2,female_lfp,94.2
9,Q2,compl_uppersec_f,34.1


## 8. How to use `all_data.csv`

```python
import pandas as pd
df = pd.read_csv("../data/all_data.csv")       # from a notebook in analysis/
dev = df[df.developing]                          # low + middle income only
sa  = df[df.south_asia]                          # traditional South Asia (8 countries)
```

**Things to know**

1. **UNESCO columns are only filled for developing countries.** High-income countries are in the file (for comparison with the World Bank/ITU variables) but have `NaN` for every UNESCO variable.
2. **Completion rates come in two versions.** `compl_*` are survey-based (sparse, ~1,000 country-years); `compl_*_mod` are UNESCO's modelled estimates (complete annual series). The modelled ones are easier for time trends; the survey ones are closer to raw data.
3. **Education spending appears twice:** `edu_spend_gdp` (World Bank) and `edu_spend_pct_gdp` (UNESCO) are the same indicator from two sources (correlation ≈ 0.99). Pick one.
4. **`income_group` and `developing` use today's classification** for every year.
5. **Use `south_asia`, not `region`,** for South Asia comparisons (the World Bank region excludes Afghanistan and Pakistan).
6. **Not yet merged:** UNDP Gender Inequality Index (Q2 proposal). Add it by saving a cleaned `iso3`/`year` CSV in `data/` and adding a merge step in section 4 plus its codebook rows in section 5.